# 1 - Colorectal fm local validation

In [1]:
import digitalhub as dh

In [2]:
project = dh.get_or_create_project('colorectal-fm-1')

## Hugging Face credentials

CONCH v1.5 is gated. First request access at https://huggingface.co/MahmoodLab/conchv1_5, then run the next cell once to store a read token as a DigitalHub secret. The masked prompt keeps the token out of the notebook.

In [ ]:
# from getpass import getpass

# hf_token = getpass("Hugging Face read token: ")
# hf_token_secret = project.new_secret(name="HF_TOKEN", secret_value=hf_token)
# del hf_token

In [3]:
hf_token_secret = project.get_secret("HF_TOKEN")

> Log them directly from the workspace (first cell) or get if they are already loaded (second cell)

## Log input files

Labels csv

In [ ]:
crs1_labels_data = project.new_artifact(
    name="crs1_labels",
    kind="artifact",
    path="s3://dsh-datalake/colorectal-fm-1/IMP-CRS2024-Dataset/CRS1/labels.csv",
)

In [4]:
crs1_labels_data = project.get_artifact('crs1_labels')
crs1_labels_data.key

'store://colorectal-fm-1/artifact/artifact/crs1_labels:f8ea5425eb5440af8ee68f25c5caa263'

In [13]:
crs2_labels_data = project.new_artifact(
    name="crs2_labels",
    kind="artifact",
    path="s3://dsh-datalake/colorectal-fm-1/IMP-CRS2024-Dataset/CRS2/labels.csv",
)

In [ ]:
crs2_labels_data = project.get_artifact('crs2_labels')
crs2_labels_data.key

In [14]:
crst_labels_data = project.new_artifact(
    name="crst_labels",
    kind="artifact",
    path="s3://dsh-datalake/colorectal-fm-1/IMP-CRS2024-Dataset/CRS_Test/labels.csv",
)

In [ ]:
crst_labels_data = project.get_artifact('crst_labels')
crst_labels_data.key

Slides files

In [ ]:
crs1_slides_data = project.new_artifact(
    name="crs1_slides",
    kind="artifact",
    path="s3://dsh-datalake/colorectal-fm-1/IMP-CRS2024-Dataset/CRS1/slides/",
)

In [5]:
crs1_slides_data = project.get_artifact('crs1_slides')
crs1_slides_data.key

'store://colorectal-fm-1/artifact/artifact/crs1_slides:6eb2e1f0012f4762b4c24073dc14fe2d'

In [15]:
crs2_slides_data = project.new_artifact(
    name="crs2_slides",
    kind="artifact",
    path="s3://dsh-datalake/colorectal-fm-1/IMP-CRS2024-Dataset/CRS2/slides/",
)

In [ ]:
crs2_slides_data = project.get_artifact('crs2_slides')
crs2_slides_data.key

In [16]:
crst_slides_data = project.new_artifact(
    name="crst_slides",
    kind="artifact",
    path="s3://dsh-datalake/colorectal-fm-1/IMP-CRS2024-Dataset/CRS_Test/slides/",
)

In [ ]:
crst_slides_data = project.get_artifact('crst_slides')
crst_slides_data.key

## Declaration

Declare the function...

In [ ]:
trident_function = project.new_function(
    name="1-trident-feature-extraction",
    kind="python",
    python_version="PYTHON3_11",
    requirements="1-trident/requirements.txt",
    code_src="1-trident",
    handler="main:trident_feature_extraction",
    # image=trident_image_ref
)

... or retrieve an already existing function version by supplying its entity ID

In [6]:
# Replace <entity-id> with the ID printed when the function is created, then uncomment.
trident_function = project.get_function("1-trident-feature-extraction",
    # entity_id=""
)

## Building

Build the function

In [ ]:
trident_build = trident_function.run(action="build",
    instructions=[
        "apt-get update && apt-get install -y git libopenslide0 libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*"
    ],
    resources={"cpu": "8", "mem": "16Gi", "disk": "20Gi"},
)

Get the reference of the function

In [7]:
trident_image_ref = trident_function.refresh().spec.image
print(trident_image_ref)

harbor.atlas.fbk.eu/dsh/dsh-platform-colorectal-fm-1-1-trident-feature-extraction:28a2f


## Running

Run the function

In [8]:
extraction_preset = "conch"
extraction_presets = {
    "conch": {"patch_encoder": "conch_v15", "slide_encoder": None, "mag": 20, "patch_size": 512},
    "prism": {"patch_encoder": "virchow", "slide_encoder": "prism", "mag": 20, "patch_size": 224},
    "titan": {"patch_encoder": "conch_v15", "slide_encoder": "titan", "mag": 20, "patch_size": 512},
}
extraction_parameters = extraction_presets[extraction_preset]

# Corresponding feature directories:
# CONCH: 20x_512px_0px_overlap/features_conch_v15
# PRISM: 20x_224px_0px_overlap/slide_features_prism
# TITAN: 20x_512px_0px_overlap/slide_features_titan


In [ ]:
dh.get_k8s_resource_profiles()

In [11]:
trident_run = trident_function.run(action="job",
    inputs={
        "slides": crs1_slides_data.key,
        "labels": crs1_labels_data.key,
    },
    parameters={
        "output_artifact_name": "1-trident-features-val",
        "task": "all",
        **extraction_parameters,
        "gpus": [0],
    },
    profile="1xrtx6000pro",
    resources={"cpu": "8", "mem": "32Gi", "gpu": "1", "disk": "1200Gi"},  # adjust disk to the slide set
    secrets=["HF_TOKEN"],
    local_execution=False,
)

In [17]:
trident_run = trident_function.run(action="job",
    inputs={
        "slides": crs2_slides_data.key,
        "labels": crs2_labels_data.key,
    },
    parameters={
        "output_artifact_name": "1-trident-features-train",
        "task": "all",
        **extraction_parameters,
        "gpus": [0],
    },
    profile="1xrtx6000pro",
    resources={"cpu": "8", "mem": "32Gi", "gpu": "1", "disk": "3300Gi"},  # adjust disk to the slide set
    secrets=["HF_TOKEN"],
    local_execution=False,
)

In [18]:
trident_run = trident_function.run(action="job",
    inputs={
        "slides": crst_slides_data.key,
        "labels": crst_labels_data.key,
    },
    parameters={
        "output_artifact_name": "1-trident-features-internal-test",
        "task": "all",
        **extraction_parameters,
        "gpus": [0],
    },
    profile="1xrtx6000pro",
    resources={"cpu": "8", "mem": "32Gi", "gpu": "1", "disk": "1000Gi"},  # adjust disk to the slide set
    secrets=["HF_TOKEN"],
    local_execution=False,
)

In [ ]:
# trident_run = trident_function.run(action="job",
#     inputs={
#         "slides": slides_data.key,
#         "labels": labels_data.key,
#     },
#     parameters={
#         "output_artifact_name": "1-trident-features-external-test",
#         "task": "all",
#         **extraction_parameters,
#         "gpus": [0, 1],
#     },
#     profile="1xrtx6000pro",
#     resources=actual_resources,  # adjust disk to the slide set
#     secrets=["HF_TOKEN"],
#     local_execution=False,
# )